# Week 5 & 6 Deliverables   

## 1. Download Data

### Samples
- [Short-read Illumina (**interleaved** paired-end FASTQ)](https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2)
- [Long-read PacBio](https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2)

### Reference Genome
The hg38 (or GRCh38) version of the human genome, focusing on the chromosome that contains these genes ([chromosome 10](https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz)): 
- CYP2C8 (regulates many drugs, including anticancer, diabetes and blood pressure drugs)
- CYP2C9 (regulates many common drugs, including warfarin / Coumadin and NSAIDs such as Advil)
- CYP2C19 (regulates… yup, many common drugs, including antiplatelet drugs, antidepressants and anti-epileptic drugs).            

|Genes | CYP2C8 | CYP2C9 | CYP2C19 |
| --- | --- | --- | ---|
| Genomic sequence | chr10:95036772-95069497 | chr10:94938658-94990091 | chr10:94762681-94855547 |
| Strand | - | + | + | 
| Genomic size | 32726 | 51434 | 92867 |

**Sources:**   

In [ ]:
!mkdir -p data

# SAMPLES
# Download Illumina and PacBio data
!wget -P data/ https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2
!wget -P data/ https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2
!bunzip2 data/*.bz2

# REFERENCE GENOME 
# chr10 containing CYP2C genes
!wget -P data/ https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz
!gunzip data/*.gz

In [ ]:
# Locate the CYP2C8, CYP2C9, and CYP2C19 genes in the reference genome
# only extract the necessary

import pysam

path = "data/chr10.fa"
fasta = pysam.FastaFile(path)

GENE_INFO = {
    "CYP2C19": {"chr": "chr10", "start": 94762681, "end": 94855547, "strand": "+"},
    "CYP2C9":  {"chr": "chr10", "start": 94938658, "end": 94990091, "strand": "+"},
    "CYP2C8":  {"chr": "chr10", "start": 95036772, "end": 95069497, "strand": "-"},
}

with open("data/CYP2C_extract.fa", "w") as out_f:
    for name, info in GENE_INFO.items():
        seq = fasta.fetch(info["chr"], info["start"] - 1, info["end"])
        out_f.write(f">{name} {info['chr']}:{info['start']}-{info['end']} ({info['strand']})\n")
        out_f.write(seq + "\n")

fasta.close()

In [ ]:
# REFERENCE GENOME 
# chr10 containing CYP2C genes
# Downloading important genes 

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import requests

# Output FASTA file
output_file = "data/reference_genome.fa"

# Gene coordinates (hg38, UCSC)
GENE_INFO = {
    "CYP2C19": {"chr": "chr10", "start": 94762681, "end": 94855547, "strand": "+"},
    "CYP2C9":  {"chr": "chr10", "start": 94938658, "end": 94990091, "strand": "+"},
    "CYP2C8":  {"chr": "chr10", "start": 95036772, "end": 95069497, "strand": "-"},
}

# UCSC FASTA API template
ucsc_fasta_url = "https://api.genome.ucsc.edu/getData/sequence?genome=hg38;chrom={chr};start={start};end={end}"

records = []

for gene, info in GENE_INFO.items():
    url = ucsc_fasta_url.format(chr=info["chr"], start=info["start"]-1, end=info["end"])
    r = requests.get(url)
    r.raise_for_status()
    seq = r.json()["dna"]

    # Create SeqRecord
    record = SeqRecord(
        Seq(seq),
        id=gene,
        description=f"{info['chr']}:{info['start']}-{info['end']} ({info['strand']})"
    )
    records.append(record)

# Write all genes to a single FASTA
with open(output_file, "w") as f:
    SeqIO.write(records, f, "fasta")

## 2. Align Samples to Reference Genome

**Short-read Illumina (interleaved paired-end FASTQ)**  
- -x: applies multiple options at the same time 
- sr: short read alignment without slicing

**Long-read PacBio**  
- map-hifi: align PacBio high-fidelity reads to a reference genome


In [ ]:
# minimap index
!minimap2 -d data/reference_genome.mmi data/reference_genome.fa 

# Short read Illumina 
!minimap2 -ax sr data/reference_genome.mmi data/illumina.fq > data/illumina.sam 

# Long read BioPac
!minimap2 -ax map-hifi data/reference_genome.mmi data/pacbio.fq > data/pacbio.sam

In [ ]:
# Method 2

# minimap index
!minimap2 -d data/CYP2C_extract.mmi data/CYP2C_extract.fa

# Short read Illumina 
!minimap2 -ax sr data/CYP2C_extract.mmi data/illumina.fq > data/illumina_extract.sam 

# Long read BioPac
!minimap2 -ax map-hifi data/CYP2C_extract.mmi data/pacbio.fq > data/pacbio_extract.sam

In [ ]:
# Convert SAM to sorted BAM
!samtools view -bS data/illumina.sam | samtools sort -o data/illumina.bam
!samtools view -bS data/pacbio.sam | samtools sort -o data/pacbio.bam

# Index BAM for random access
!samtools index -b data/illumina.bam
!samtools index -b data/pacbio.bam

In [ ]:
# Method 2

# Convert SAM to sorted BAM
!samtools view -bS data/illumina_extract.sam | samtools sort -o data/illumina_extract.bam
!samtools view -bS data/pacbio_extract.sam | samtools sort -o data/pacbio_extract.bam

# Index BAM for random access
!samtools index -b data/illumina_extract.bam
!samtools index -b data/pacbio_extract.bam

## 3. Variant Calling
bcftools mpileup -f reference.fa alignments.bam | bcftools call -mv -Ob -o calls.bcf    
- mpileup part generates genotype likelihoods at each genomic position with coverage
- call part makes the actual calls
- -m switch tells the program to use the default calling method
- -v option asks to output only variant sites
- -O option selects the output format

bcftools mpileup -Ou -f reference.fa alignments.bam | bcftools call -mv -Ob -o calls.bcf
- Do not waste computer’s time by making mpileup convert from the internal binary representation (BCF) to text (VCF), only to be immediately converted back to binary representation by call. Instead, use -Ou to work with uncompressed BCF output

In [ ]:
# Index reference genome
!samtools faidx data/reference_genome.fa

# Method 2
!samtools faidx data/CYP2C_extract.fa

In [ ]:
# Call variants 
!bcftools mpileup -Ou -f data/reference_genome.fa data/illumina.bam | bcftools call -mv --ploidy 2 -Oz -o data/illumina.vcf.gz
!bcftools index data/illumina.vcf.gz

!bcftools mpileup -Ou -f data/reference_genome.fa data/pacbio.bam | bcftools call -mv --ploidy 2 -Oz -o data/pacbio.vcf.gz
!bcftools index data/pacbio.vcf.gz

!bcftools convert -O v data/illumina.vcf.gz > data/illumina.vcf
!bcftools convert -O v data/pacbio.vcf.gz > data/pacbio.vcf

In [ ]:
# Method 2

# Call variants 
!bcftools mpileup -Ou -f data/CYP2C_extract.fa data/illumina_extract.bam | bcftools call -mv --ploidy 2 -Oz -o data/illumina_extract.vcf.gz
!bcftools index data/illumina_extract.vcf.gz

!bcftools mpileup -Ou -f data/CYP2C_extract.fa data/pacbio_extract.bam | bcftools call -mv --ploidy 2 -Oz -o data/pacbio_extract.vcf.gz
!bcftools index data/pacbio_extract.vcf.gz

!bcftools convert -O v data/illumina_extract.vcf.gz > data/illumina_extract.vcf
!bcftools convert -O v data/pacbio_extract.vcf.gz > data/pacbio_extract.vcf

## 4. Phase Variant VCFs 

In [1]:
!extractHAIRS --bam data/illumina.bam --VCF data/illumina.vcf --out data/illumina.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/illumina.fragments --VCF data/illumina.vcf --output data/illumina.hapcut

!extractHAIRS --pacbio 1 --bam data/pacbio.bam --VCF data/pacbio.vcf --out data/pacbio.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/pacbio.fragments --VCF data/pacbio.vcf --output data/pacbio.hapcut


Extracting haplotype informative reads from bamfiles data/illumina.bam minQV 13 minMQ 20 maxIS 1000 

VCF file data/illumina.vcf has 2142 variants 
adding chrom CYP2C19 to index 
adding chrom CYP2C9 to index 
adding chrom CYP2C8 to index 
vcffile data/illumina.vcf chromosomes 3 hetvariants 2012 variants 2142 
detected 9 variants with two non-reference alleles, these variants will not be phased

##########################################################################################
3 chromosomes/contigs detected in input VCF file
processing each contig separately using --regions option will be more efficient for large genomes
############################################################################################

reading fasta index file data/reference_genome.fa.fai ... fasta file data/reference_genome.fa has 3 chromosomes/contigs

found match for reference contig CYP2C19 in VCF file index 
contig CYP2C19 length 92867
found match for reference contig CYP2C9 in VCF file index 
c

In [ ]:
# Method 2
!extractHAIRS --bam data/illumina_extract.bam --VCF data/illumina_extract.vcf --out data/illumina_extract.fragments --ref data/CYP2C_extract.fa
!HAPCUT2 --fragments data/illumina_extract.fragments --VCF data/illumina_extract.vcf --output data/illumina_extract.hapcut

!extractHAIRS --pacbio 1 --bam data/pacbio_extract.bam --VCF data/pacbio_extract.vcf --out data/pacbio_extract.fragments --ref data/CYP2C_extract.fa
!HAPCUT2 --fragments data/pacbio_extract.fragments --VCF data/pacbio_extract.vcf --output data/pacbio_extract.hapcut

## 5. Variant Analysis
- Now you should have two phased VCF files (one for each sequencing technology). Compare these VCFs. 
    - How many variants are shared between the VCFs? How many are not?
- Select 2-3 variants that are not common (if any) and check which technology supports this variant. Open both BAM files in IGV and take a screenshot of each problematic discordant location. What can you deduce from these screenshots—are these variants sequencing-related artifacts or are they indeed true variants? Do this analysis for every gene.
- IGV screenshots can also be automated (it is a bit tricky, though—ask your LLM for help). You can opt out of doing this, but you will lose half a point.

- Expected output: Jupyter cell(s) with IGV screenshots and a discussion.

In [ ]:
!bcftools isec data/illumina.vcf.gz data/pacbio.vcf.gz

In [ ]:
# Parse VCFs
from cyvcf2 import VCF
import pandas as pd

# Load phased VCFs
illumina_path = "data/illumina.hapcut.phased.VCF"
pacbio_path = "data/pacbio.hapcut.phased.VCF"

def read_variants(path):
    variants = {}
    vcf = VCF(path)
    for v in vcf:
        key = (v.CHROM, v.POS, v.REF, tuple(v.ALT))

        # String representation of genotypes
        gt = v.gt_bases[0] if v.gt_bases is not None else None

        # Depth value 
        dp = v.INFO.get("DP", None)

        pq = v.format("PQ")[0][0] if "PQ" in v.FORMAT else None

        variants[key] = {
            "CHROM": v.CHROM,
            "POS": v.POS,
            "REF": v.REF,
            "ALT": ",".join(v.ALT),
            "QUAL": v.QUAL,
            "DP": dp,
            "GT": gt,
            "PQ": pq
        }
    return variants

illumina_var = read_variants(illumina_path)
pacbio_var   = read_variants(pacbio_path)

illumina_var_key = set(illumina_var.keys())
pacbio_var_key   = set(pacbio_var.keys())

shared_keys   = illumina_var_key & pacbio_var_key
illumina_only = illumina_var_key - pacbio_var_key
pacbio_only   = pacbio_var_key - illumina_var_key

print(f"Shared variants: {len(shared_keys)}")
print(f"Unique to Illumina: {len(illumina_only)}")
print(f"Unique to PacBio:  {len(pacbio_only)}\n")

for key in list(illumina_only)[:3]:
    v = illumina_var[key]
    print(f"[Illumina-only] {v['CHROM']}:{v['POS']} {v['REF']}->{v['ALT']}  GT={v['GT']}  DP={v['DP']}  PQ={v['PQ']}")

for key in list(pacbio_only)[:3]:
    v = pacbio_var[key]
    print(f"[PacBio-only]  {v['CHROM']}:{v['POS']} {v['REF']}->{v['ALT']}  GT={v['GT']}  DP={v['DP']}  PQ={v['PQ']}")

Shared variants: 771
Unique to Illumina: 1371
Unique to PacBio:  1234

=== Example Discordant Variants ===
[Illumina-only] CYP2C19:16128 A->G  GT=G|A  DP=94  PQ=100
[Illumina-only] CYP2C9:20382 G->A  GT=G|A  DP=124  PQ=100
[Illumina-only] CYP2C19:56829 A->G  GT=A|G  DP=209  PQ=100
[PacBio-only]  CYP2C19:32690 A->C  GT=C|A  DP=14  PQ=100
[PacBio-only]  CYP2C19:73771 A->G  GT=A|G  DP=41  PQ=100
[PacBio-only]  CYP2C19:64615 A->G  GT=A|G  DP=17  PQ=100


## 6. Star-Allele Calls
- Can you figure out the star-allele for each gene of interest? The star-allele database can be found in PharmVar; see this for CYP2C19. Your answer should be something like CYP2C19*12 because X, Y and Z. This step does not have to be automated, but should be at least explained in the notebook.
    - Hint: use phased data!

- Expected output: Jupyter cell(s) with discussion (and code, if you want to do it that way).